# Finly — Fase 4: Chat com os Dados em Linguagem Natural

Digite uma pergunta em português e o Llama 3.1 gera e executa o código Python automaticamente.

Exemplos:
- *"Quanto gastei no total com delivery?"*
- *"Qual foi meu maior gasto do ano?"*
- *"Em qual mês gastei mais dinheiro?"*
- *"Qual dia da semana eu gasto mais com lazer?"*

## 1. Imports e carregamento dos dados

In [17]:
import pandas as pd
import numpy as np
import requests
import json

OLLAMA_URL = "http://localhost:11434/api/generate"
MODELO     = "llama3.1"

df = pd.read_csv("../extrato_categorizado.csv", parse_dates=["data"])
df["dia_semana"]    = df["data"].dt.day_name(locale="pt_BR").str.capitalize()
df["mes"]           = df["data"].dt.month
df["mes_nome"]      = df["data"].dt.strftime("%b").str.capitalize()
df["semana_do_mes"] = ((df["data"].dt.day - 1) // 7) + 1
df["ano_mes"]       = df["data"].dt.to_period("M").astype(str)

print(f"Total de transacoes: {len(df)}")
print(f"Categorias: {sorted(df['categoria'].unique())}")

Total de transacoes: 459
Categorias: ['alimentação', 'casa', 'delivery', 'educação', 'lazer', 'outros', 'saúde', 'transporte', 'vestuário']


## 2. Motor do chat

In [18]:
def gerar_codigo(pergunta):
    cats = str(sorted(df["categoria"].unique()))
    n    = len(df)

    prompt = (
        "# Contexto\n"
        f"Existe um DataFrame chamado df JA CARREGADO com {n} linhas.\n"
        "Colunas: data(datetime), descricao(str), valor(float), categoria(str),\n"
        "         mes(int 1-12), mes_nome(str), dia_semana(str), semana_do_mes(int), ano_mes(str)\n"
        f"Categorias: {cats}\n\n"
        "# Regras OBRIGATORIAS\n"
        "1. NAO escreva import. NAO crie um novo df. NAO redefina df.\n"
        "2. Use apenas df, pd e np que ja existem.\n"
        "3. Termine com print() mostrando a resposta em portugues.\n"
        "4. Escreva APENAS o codigo, sem texto, sem markdown.\n\n"
        f"# Pergunta\n{pergunta}\n\n"
        "# Codigo:\n"
    )

    payload = {
        "model": MODELO,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0, "num_predict": 250}
    }

    r = requests.post(OLLAMA_URL, json=payload, timeout=180)
    r.raise_for_status()

    codigo = r.json()["response"].strip()
    codigo = codigo.replace("```python", "").replace("```", "").strip()

    # Remove linhas que redefinem df ou importam libs
    linhas = []
    for linha in codigo.split("\n"):
        proibido = ["import ", "df = pd.DataFrame", "df = {", "df=pd.DataFrame"]
        if any(p in linha for p in proibido):
            continue
        linhas.append(linha)

    return "\n".join(linhas).strip()


def perguntar(pergunta):
    print(f"Pergunta: {pergunta}")
    print("-" * 55)
    codigo = gerar_codigo(pergunta)
    print("Codigo gerado pelo Llama:")
    print(codigo)
    print("-" * 55)
    print("Resposta:")
    try:
        exec(codigo, {"df": df, "pd": pd, "np": np, "print": print})
    except Exception as e:
        print(f"Erro ao executar: {e}")
    print()


print("Motor do chat pronto!")

Motor do chat pronto!


## 3. Perguntas de exemplo

In [20]:
perguntar("Quanto gastei no total com delivery?")

Pergunta: Quanto gastei no total com delivery?
-------------------------------------------------------
Codigo gerado pelo Llama:
gastos_delivery = df[df['categoria'] == 'delivery']['valor'].sum()
print(f'Você gastou R${gastos_delivery:.2f} no total com delivery.')
-------------------------------------------------------
Resposta:
Você gastou R$6568.75 no total com delivery.



In [21]:
perguntar("Qual foi meu maior gasto do ano e em qual categoria?")

Pergunta: Qual foi meu maior gasto do ano e em qual categoria?
-------------------------------------------------------
Codigo gerado pelo Llama:
max_gasto = df.groupby('categoria')['valor'].sum().idxmax()
print(f'O maior gasto do ano foi de {df.loc[df["categoria"] == max_gasto, "valor"].sum():.2f} reais na categoria {max_gasto}.')
-------------------------------------------------------
Resposta:
O maior gasto do ano foi de 20038.70 reais na categoria alimentação.



In [22]:
perguntar("Em qual mes gastei mais dinheiro?")

Pergunta: Em qual mes gastei mais dinheiro?
-------------------------------------------------------
Codigo gerado pelo Llama:
df['ano_mes'] = df['data'].dt.strftime('%Y-%m')
meses_gastos = df.groupby('ano_mes')['valor'].sum().reset_index()
maior_mesa = meses_gastos.loc[meses_gastos['valor'].idxmax()]['ano_mes']
print(f'O mês em que você gastou mais dinheiro foi o {pd.to_datetime(maior_mesa).strftime('%B de %Y')}.')
-------------------------------------------------------
Resposta:
O mês em que você gastou mais dinheiro foi o November de 2024.



In [25]:
perguntar("Qual dia da semana eu gasto mais com lazer?")

Pergunta: Qual dia da semana eu gasto mais com lazer?
-------------------------------------------------------
Codigo gerado pelo Llama:
df['dia_semana'] = pd.to_datetime(df['data']).dt.day_name()
lazer_df = df[df['categoria'] == 'lazer']
mais_gastado = lazercf.groupby('dia_semana')['valor'].sum().idxmax()
print(f'O dia da semana em que você gasta mais com lazer é {mais_gastado}.')
-------------------------------------------------------
Resposta:
Erro ao executar: name 'lazercf' is not defined



In [24]:
perguntar("Quanto gastei em media por mes com transporte?")

Pergunta: Quanto gastei em media por mes com transporte?
-------------------------------------------------------
Codigo gerado pelo Llama:
media_gasto_transporte = df[df['categoria'] == 'transporte']['valor'].mean()
print(f'Media de gastos com transporte: R${media_gasto_transporte:.2f}')
-------------------------------------------------------
Resposta:
Media de gastos com transporte: R$145.52



## 4. Sua pergunta aqui

In [19]:
# Troque o texto abaixo e rode!
perguntar("Qual categoria cresceu mais do primeiro para o segundo semestre?")

Pergunta: Qual categoria cresceu mais do primeiro para o segundo semestre?
-------------------------------------------------------
Codigo gerado pelo Llama:
df_semestre1 = df[df['mes'] <= 6]
df_semestre2 = df[df['mes'] > 6]

crescimento_categoria = (df_semestre2.groupby('categoria')['valor'].sum() / 
                         df_semestre1.groupby('categoria')['valor'].sum()).reset_index()

maior_crescimento = crescimento_categoria.loc[crescimento_categoria['valor'].idxmax()]

print(f'A categoria que cresceu mais do primeiro para o segundo semestre é: {maior_crescimento["categoria"]} com um aumento de {maior_crescimento["valor"]:.2f}%.')
-------------------------------------------------------
Resposta:
A categoria que cresceu mais do primeiro para o segundo semestre é: vestuário com um aumento de 3.75%.

